# KS Fotoğraf → GLB
Ücretsiz Google Colab GPU üzerinde açık kaynak TripoSG ile GLB üretir. Üst menüden **Çalışma zamanı → Tümünü çalıştır** seçin. Kurulum otomatik yapılır; alttaki form açılınca fotoğrafı yükleyin.

In [ ]:
#@title KS 3D Üreticiyi Başlat { display-mode: "form" }
import os, sys, subprocess, uuid
from pathlib import Path
from IPython.display import display, HTML
display(HTML('<h3>KS 3D motoru hazırlanıyor…</h3><p>İlk açılış model indirmesi nedeniyle birkaç dakika sürebilir.</p>'))
if subprocess.run(['nvidia-smi'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode != 0:
    raise RuntimeError('GPU seçili değil. Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seçin.')
root=Path('/content/TripoSG')
if not root.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/VAST-AI-Research/TripoSG.git',str(root)],check=True,stdout=subprocess.DEVNULL)
req=root/'requirements-colab.txt'
req.write_text('\n'.join(x for x in (root/'requirements.txt').read_text().splitlines() if not x.startswith('numpy==')))
subprocess.run([sys.executable,'-m','pip','install','-q','numpy<2','-r',str(req),'rembg[cpu]','gradio>=5,<7'],check=True)
sys.path.insert(0,str(root));sys.path.insert(0,str(root/'scripts'))
import torch, numpy as np, trimesh, gradio as gr
from PIL import Image
from rembg import remove, new_session
from huggingface_hub import snapshot_download
from triposg.pipelines.pipeline_triposg import TripoSGPipeline
weights=root/'pretrained_weights/TripoSG'
snapshot_download(repo_id='VAST-AI/TripoSG',local_dir=str(weights))
pipe=TripoSGPipeline.from_pretrained(str(weights)).to('cuda',torch.float16)
rmbg=new_session('u2netp')
def prepare(image):
    import io
    raw=io.BytesIO();Image.fromarray(image).save(raw,format='PNG')
    rgba=Image.open(io.BytesIO(remove(raw.getvalue(),session=rmbg))).convert('RGBA')
    bbox=rgba.getchannel('A').getbbox()
    if bbox: rgba=rgba.crop(bbox)
    w,h=rgba.size;side=max(w,h);pad=max(12,int(side*.12));size=side+2*pad
    canvas=Image.new('RGBA',(size,size),(255,255,255,0));canvas.alpha_composite(rgba,((size-w)//2,(size-h)//2))
    result=Image.new('RGB',canvas.size,'white');result.paste(canvas.convert('RGB'),mask=canvas.getchannel('A'));return result
def generate(image,quality,seed,progress=gr.Progress()):
    if image is None: raise gr.Error('Önce fotoğraf yükleyin.')
    progress(.1,desc='Arka plan hazırlanıyor');img=prepare(image)
    steps,faces=(35,120000) if quality=='Hızlı' else (50,250000)
    progress(.25,desc='3D geometri üretiliyor')
    with torch.no_grad(): out=pipe(image=img,generator=torch.Generator(device='cuda').manual_seed(int(seed)),num_inference_steps=steps,guidance_scale=7.0).samples[0]
    mesh=trimesh.Trimesh(out[0].astype(np.float32),np.ascontiguousarray(out[1]),process=False)
    if len(mesh.faces)>faces:
        try: mesh=mesh.simplify_quadric_decimation(face_count=faces)
        except Exception: pass
    output=f'/content/KS_OUTPUT_{uuid.uuid4().hex[:8]}.glb';mesh.export(output);torch.cuda.empty_cache();progress(1,desc='GLB hazır');return output,output
with gr.Blocks(title='KS Fotoğraf → GLB') as app:
    gr.Markdown('## KS Fotoğraf → GLB\nHavacılık aracını mümkünse sade arka plan önünde, tamamı görünecek şekilde yükleyin.')
    with gr.Row():
        image=gr.Image(label='Araç fotoğrafı',type='numpy')
        preview=gr.Model3D(label='Üretilen 3D model')
    with gr.Row(): quality=gr.Radio(['Hızlı','Kaliteli'],value='Kaliteli',label='Kalite');seed=gr.Number(value=42,precision=0,label='Varyasyon')
    button=gr.Button('3D MODELİ ÜRET',variant='primary');download=gr.File(label='GLB dosyasını indir')
    button.click(generate,[image,quality,seed],[preview,download])
app.launch(share=True,debug=False)
